## Dependencies

In [3]:
!pip install -qU "langchain[aws]" pinecone langchain-pinecone requests langchain-text-splitters

In [9]:
release_notes_2025 = "https://docs.pinecone.io/release-notes/2025.md"
release_notes_2024 = "https://docs.pinecone.io/release-notes/2024.md"

import requests
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on = [("#", "release"), ("##", "month_year"), ("###", "feature")]
)

def download_content(url):
    response = requests.get(url)
    return response.text

def add_document_metadata(doc, new_metadata):
    old_metadata = doc.metadata
    new_metadata = {**old_metadata, **new_metadata}
    return Document(page_content = doc.page_content, metadata = new_metadata)



for url in [release_notes_2025, release_notes_2024]:
    text = download_content(url)
    split_text = splitter.split_text(text)
    split_text = [
        add_document_metadata(doc, {"source": url,"chunk_num": num})
        for num, doc in enumerate(split_text)
        ]

print(split_text)



[Document(metadata={'source': 'https://docs.pinecone.io/release-notes/2024.md', 'chunk_num': 0}, page_content='> ## Documentation Index\n> Fetch the complete documentation index at: https://docs.pinecone.io/llms.txt\n> Use this file to discover all available pages before exploring further.'), Document(metadata={'release': '2024 releases', 'source': 'https://docs.pinecone.io/release-notes/2024.md', 'chunk_num': 1}, page_content='> Pinecone release notes — 2024 releases:'), Document(metadata={'release': '2024 releases', 'month_year': 'December 2024', 'source': 'https://docs.pinecone.io/release-notes/2024.md', 'chunk_num': 2}, page_content='<Update label="2024-12-23" tags={["Database"]}>'), Document(metadata={'release': '2024 releases', 'month_year': 'December 2024', 'feature': 'Increased namespaces limit', 'source': 'https://docs.pinecone.io/release-notes/2024.md', 'chunk_num': 3}, page_content='Customers on the [Standard plan](https://www.pinecone.io/pricing/) can now have up to 25,000 

In [10]:
import os
import dotenv
dotenv.load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

In [ ]:
from pinecone import Pinecone

pinecone = Pinecone(
    api_key = PINECONE_API_KEY
)

In [15]:
from pinecone import ServerlessSpec

index_name = "release-update-demo"

if not pinecone.has_index(index_name):
    pinecone.create_index(
        name = index_name,
        dimension = 1536,
        metric = "cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pinecone.Index(name=index_name)

index.describe_index_stats()

c:\Users\RichardHawkins\Revature\2370-2371-DF-AI\trainer-code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

In [16]:
from langchain_aws import BedrockEmbeddings

embeddings = BedrockEmbeddings(
    model_id = "amazon.titan-embed-text-v1",
    region_name = "us-east-1"
)

In [17]:
from langchain_pinecone import PineconeVectorStore
vector_store = PineconeVectorStore(index=index, embedding=embeddings)

In [18]:
def scrub_url(url):
    return url.split("/")[-1].replace(".md", "")

def gen_id(doc_chunk):
    title = scrub_url(doc_chunk.metadata["source"])
    chunk_num = doc_chunk.metadata["chunk_num"]
    feature = doc_chunk.metadata["feature"] if "feature" in doc_chunk.metadata else "na"
    return f"#release_{title}#feature_{feature}#chunk_num{chunk_num}"

ids = [gen_id(doc_chunk) for doc_chunk in split_text]

vector_store.add_documents(documents=split_text, ids=ids)

['#release_2024#feature_na#chunk_num0',
 '#release_2024#feature_na#chunk_num1',
 '#release_2024#feature_na#chunk_num2',
 '#release_2024#feature_Increased namespaces limit#chunk_num3',
 '#release_2024#feature_Pinecone Assistant JSON mode and EU region deployment#chunk_num4',
 '#release_2024#feature_Released Spark-Pinecone connector v1.2.0#chunk_num5',
 '#release_2024#feature_New integration with HoneyHive#chunk_num6',
 '#release_2024#feature_Released Python SDK v5.4.2#chunk_num7',
 '#release_2024#feature_Launch week: Pinecone Local#chunk_num8',
 '#release_2024#feature_Launch week: Enhanced security and access controls#chunk_num9',
 '#release_2024#feature_Launch week: `pinecone-rerank-v0` and `cohere-rerank-3.5` on Pinecone Inference#chunk_num10',
 '#release_2024#feature_Launch week: Integrated Inference#chunk_num11',
 '#release_2024#feature_Released .NET SDK v2.1.0#chunk_num12',
 '#release_2024#feature_Improved batch deletion guidance#chunk_num13',
 '#release_2024#feature_Launch week: R

In [19]:
from langchain_aws import ChatBedrock
llm = ChatBedrock(
    model_id = "anthropic.claude-3-haiku-20240307-v1:0",
    region_name = "us-east-1", 
    model_kwargs={
        "temperature": 0.1
    }
)

In [26]:
query = "tell me about the latest version of the pinecone python sdk"

print(llm.invoke(query).content)

The latest version of the Pinecone Python SDK is 2.1.0, which was released on May 11, 2023.

Here are some key details about the latest version:

1. **Improved Performance**: The 2.1.0 release includes several performance improvements, such as faster vector upserts and query operations.

2. **New Features**:
   - **Batch Upsert**: The SDK now supports batch upsert operations, allowing you to upload multiple vectors at once, improving efficiency.
   - **Async Support**: The SDK now includes asynchronous versions of many operations, enabling you to perform tasks concurrently and improve overall throughput.
   - **Metadata Filtering**: You can now filter your vector search results based on metadata, providing more granular control over the data you retrieve.

3. **Bug Fixes**: The 2.1.0 release includes several bug fixes and stability improvements, addressing issues reported by the community.

4. **Documentation Updates**: The SDK documentation has been updated to reflect the new features

In [27]:
retrieved = vector_store.similarity_search(query, k=5)
doc_content = "\n\n".join(doc.page_content for doc in retrieved)
print(doc_content)

Released [`v4.0.0`](https://github.com/pinecone-io/pinecone-ts-client/releases/tag/v4.0.0) of the [Pinecone Node.js SDK](/reference/sdks/node/overview). This version uses the latest stable API version, `2024-10`, and adds support for [reranking](/guides/search/rerank-results) and [import](/guides/index-data/import-data).  
***  
Released [`v2.0.0`](https://github.com/pinecone-io/go-pinecone/releases/tag/v2.0.0) of the [Pinecone Go SDK](/reference/sdks/go/overview). This version uses the latest stable API version, `2024-10`, and adds support for [reranking](/guides/search/rerank-results) and [import](/guides/index-data/import-data).  
***  
Released [`v3.0.0`](https://github.com/pinecone-io/pinecone-java-client/releases/tag/v3.0.0) of the [Pinecone Java SDK](/reference/sdks/java/overview). This version uses the latest stable API version, `2024-10`, and adds support for [embedding](/reference/api/2025-01/inference/generate-embeddings), [reranking](/reference/api/2025-01/inference/rerank)

In [28]:
prompt = f""" You are an assistant that answers questions exclusively about the Pinecone SDK release notes.property

Here's the question:
{query}

Here's some documents from the release notes to help answer the question:
{doc_content}

Answer:
"""
print(prompt)

 You are an assistant that answers questions exclusively about the Pinecone SDK release notes.property

Here's the question:
tell me about the latest version of the pinecone python sdk

Here's some documents from the release notes to help answer the question:
Released [`v4.0.0`](https://github.com/pinecone-io/pinecone-ts-client/releases/tag/v4.0.0) of the [Pinecone Node.js SDK](/reference/sdks/node/overview). This version uses the latest stable API version, `2024-10`, and adds support for [reranking](/guides/search/rerank-results) and [import](/guides/index-data/import-data).  
***  
Released [`v2.0.0`](https://github.com/pinecone-io/go-pinecone/releases/tag/v2.0.0) of the [Pinecone Go SDK](/reference/sdks/go/overview). This version uses the latest stable API version, `2024-10`, and adds support for [reranking](/guides/search/rerank-results) and [import](/guides/index-data/import-data).  
***  
Released [`v3.0.0`](https://github.com/pinecone-io/pinecone-java-client/releases/tag/v3.0.0)

In [29]:
answer = llm.invoke(prompt)
print(answer.content)

Based on the release notes provided, here are the key details about the latest version of the Pinecone Python SDK:

- The latest version is `v5.4.2`, released on 2024-11-08.
- This version adds a required `metric` keyword argument to the `query_namespaces` method. This change enables the SDK to merge results no matter how many results are returned.
- Previous versions include:
  - `v5.4.0` and `v5.4.1`, released on 2024-09-18. These versions added a `query_namespaces` utility method to run a query in parallel across multiple namespaces and merge the result sets.
  - `v5.3.0`, released on 2024-09-18, which added support for import operations (in public preview).
  - `v5.3.1`, released on 2024-09-19, which added a missing `python-dateutil` dependency.
- The Pinecone Python SDK uses the latest stable API version, `2024-10`.
- Notable features added in recent versions include support for reranking documents with Pinecone Inference, Prometheus monitoring for serverless indexes, and the abil